# 04 — Vascular Feature Analysis

Compare extracted vessel morphology features across CTE stages.

**Prerequisites:** This notebook requires Phase 2 (U-Net segmentation)
and Phase 3 (feature extraction) to be complete. The cells below
show the planned analysis structure with placeholder data.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import torch
from collections import Counter


%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [4]:

COMPARISONS = {
    "control_vs_rhi": {
        "class_0": "Control",       # label in spreadsheet
        "class_1": "RHI",
        "name": "Control vs RHI",
        "description": "Effect of head impacts without CTE pathology",
    },
    "rhi_vs_low": {
        "class_0": "RHI",
        "class_1": "Low CTE",
        "name": "RHI vs Low CTE",
        "description": "Onset of early CTE changes",
    },
    "low_vs_high": {
        "class_0": "Low CTE",
        "class_1": "High CTE",
        "name": "Low CTE vs High CTE",
        "description": "Disease severity differentiation",
    },
    "control_vs_CTE": {
        "class_0": ["Control", "RHI"],
        "class_1": ["Low CTE", "High CTE"],
        "name": "No CTE vs CTE",
        "description": "Any CTE pathology vs none",
    },
}


# Label File Parsing
def load_label_file(label_file: str, comparison_key, sheet_name=0):
    df = pd.read_excel(label_file, sheet_name=sheet_name, engine="openpyxl")
    
    comp = COMPARISONS[comparison_key]

    df.columns = [
        "case_id", "age", "block_id", "stain", "description",
        "scanner", "mag", "filename", "path_group", 
        *[f"extra_{i}" for i in range(len(df.columns) - 9)]
    ]

    mask = df["path_group"].isin([comp["class_0"], comp["class_1"]])
    df = df[mask].copy()

    label_map = {comp["class_0"]: 0, comp["class_1"]: 1}
    df["label"] = df["path_group"].map(label_map)
    df["svs_stem"] = df["filename"].apply(lambda x: str(x).replace(".svs", ""))

    slide_labels = {}
    for _, row in df.iterrows():
        slide_labels[row["svs_stem"]] = (row["label"], row["case_id"])

    return slide_labels, comp



slide_labels, comp = load_label_file(label_file="/projectnb/rise2019/arushv/VascuPath/data/case_labels.xlsx", comparison_key="control_vs_rhi")
print(slide_labels)



{'110300': (1, 'K0002'), '12050': (1, 'K0008'), '34266': (1, 'K0068'), '61948': (1, 'K0091'), '10859': (1, 'K0092'), '115971': (1, 'K0095'), '10837': (1, 'K0109'), '171495': (1, 'K0116'), '168454': (1, 'K0147'), '1_23_145805': (1, 'K0159'), '21317': (1, 'K0176'), '1_18_135946': (1, 'K0197'), '67261': (1, 'K0211'), '10_21_110721': (1, 'K0214'), '69012': (1, 'K0231'), '68354': (1, 'K0243'), '171497': (1, 'K0245'), '171634': (1, 'K0260'), '62022': (1, 'K0281'), '9_19_112219': (1, 'K0330'), '1_18_125615': (1, 'K0332'), '171502': (1, 'K0339'), '10_23_111330': (1, 'K0344'), '7_22_184437': (1, 'K0348'), '2_19_150135': (1, 'K0365'), '5_16_175257': (1, 'K0370'), '2_26_135951': (1, 'K0373'), '10_6_201047': (1, 'K0379'), '171508': (1, 'K0388'), '1_18_172951': (1, 'K0395'), '2_29_134055': (1, 'K0407'), '171514': (1, 'K0413'), '6_15_154957': (1, 'K0414'), '3_20_141834': (1, 'K0415'), '5_22_103928': (1, 'K0432'), '9_18_192125': (1, 'K0436'), '11_17_214523': (1, 'K0438'), '13_8_162729': (1, 'K0442'),

In [3]:
# Training Utilities

def get_class_weights(labels):
    counts = Counter(labels)
    total = len(labels)
    weights = torch.zeros(2)
    for cls in range(2):
        if counts[cls] > 0:
            weights[cls] = total / (2.0 * counts[cls])
        else:
            weights[cls] = 1.0
    return weights
    




## Load Feature Data

Once Phase 3 is running, each slide will have a `vessel_features.csv`
in its output folder containing per-vessel measurements.

In [ ]:
# TODO: Replace with actual data loading once Phase 3 is complete
#
# Expected format:
#   import pandas as pd
#   features = pd.concat([
#       pd.read_csv(f"../outputs/{slide}/vessel_features.csv")
#       for slide in slide_names
#   ])
#
# Columns expected:
#   slide_name, cte_stage, vessel_id, diameter_mean, diameter_std,
#   tortuosity_arc_chord, branching_count, area_um2, perimeter_um,
#   wall_smoothness, x_position, y_position

# Placeholder data for notebook structure
np.random.seed(42)
n_per_group = 200

features_placeholder = {
    "control": {
        "diameter_mean": np.random.normal(45, 12, n_per_group),
        "tortuosity": np.random.normal(1.15, 0.08, n_per_group),
        "branching_density": np.random.normal(3.2, 1.1, n_per_group),
    },
    "mild_cte": {
        "diameter_mean": np.random.normal(52, 15, n_per_group),
        "tortuosity": np.random.normal(1.25, 0.12, n_per_group),
        "branching_density": np.random.normal(2.8, 1.3, n_per_group),
    },
    "severe_cte": {
        "diameter_mean": np.random.normal(60, 20, n_per_group),
        "tortuosity": np.random.normal(1.40, 0.18, n_per_group),
        "branching_density": np.random.normal(2.1, 1.5, n_per_group),
    },
}

print("Using placeholder data — replace with real features from Phase 3")

## Feature Distributions by CTE Stage

Compare vessel diameter, tortuosity, and branching density
across control, mild CTE, and severe CTE groups.

In [ ]:
groups = ["control", "mild_cte", "severe_cte"]
group_colors = ["#2ecc71", "#f39c12", "#e74c3c"]
feature_names = ["diameter_mean", "tortuosity", "branching_density"]
feature_labels = ["Mean Diameter (µm)", "Tortuosity (arc/chord)", "Branching Density (per mm²)"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for col, (feat, label) in enumerate(zip(feature_names, feature_labels)):
    data = [features_placeholder[g][feat] for g in groups]
    bp = axes[col].boxplot(data, labels=groups, patch_artist=True, widths=0.6)
    for patch, c in zip(bp["boxes"], group_colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    axes[col].set_ylabel(label)
    axes[col].set_title(feat.replace("_", " ").title())
    axes[col].tick_params(axis="x", rotation=15)

plt.suptitle("Vascular Features by CTE Stage", fontsize=14)
plt.tight_layout()
plt.show()

## Statistical Tests

Kruskal-Wallis test (non-parametric) to check if features differ
significantly across CTE stages. Bonferroni correction for
multiple comparisons.

In [ ]:
print(f"{'Feature':<25} {'H-statistic':>12} {'p-value':>12} {'p (corrected)':>15} {'Significant?':>12}")
print("-" * 80)

n_tests = len(feature_names)
alpha = 0.05

for feat, label in zip(feature_names, feature_labels):
    data = [features_placeholder[g][feat] for g in groups]
    h_stat, p_val = stats.kruskal(*data)
    p_corrected = min(p_val * n_tests, 1.0)  # Bonferroni
    sig = "Yes *" if p_corrected < alpha else "No"
    print(f"{feat:<25} {h_stat:>12.2f} {p_val:>12.6f} {p_corrected:>15.6f} {sig:>12}")

print(f"\nBonferroni-corrected alpha = {alpha}/{n_tests} = {alpha/n_tests:.4f}")

## Pairwise Comparisons

Mann-Whitney U tests between each pair of CTE stages for
significant features.

In [ ]:
from itertools import combinations

print("Pairwise Mann-Whitney U tests (Bonferroni-corrected):")
print("=" * 70)

for feat in feature_names:
    print(f"\n{feat}:")
    pairs = list(combinations(groups, 2))
    n_pairs = len(pairs)

    for g1, g2 in pairs:
        d1 = features_placeholder[g1][feat]
        d2 = features_placeholder[g2][feat]
        u_stat, p_val = stats.mannwhitneyu(d1, d2, alternative="two-sided")
        p_corr = min(p_val * n_pairs, 1.0)
        sig = "*" if p_corr < 0.05 else ""
        print(f"  {g1} vs {g2}: U={u_stat:.0f}, p={p_corr:.6f} {sig}")

## Feature Correlation Matrix

Check which vessel features correlate with each other.
Highly correlated features may be redundant for downstream
MIL classification.

In [ ]:
# Stack all features into a matrix
all_data = {}
for feat in feature_names:
    all_data[feat] = np.concatenate([features_placeholder[g][feat] for g in groups])

feat_matrix = np.column_stack([all_data[f] for f in feature_names])
corr = np.corrcoef(feat_matrix.T)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_names)))
ax.set_yticks(range(len(feature_names)))
ax.set_xticklabels(feature_labels, rotation=30, ha="right", fontsize=9)
ax.set_yticklabels(feature_labels, fontsize=9)

for i in range(len(feature_names)):
    for j in range(len(feature_names)):
        ax.text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center", fontsize=11)

plt.colorbar(im, label="Pearson r")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## Key Takeaways

- [ ] Which features differ significantly between CTE stages?
- [ ] Are the differences in the expected direction (e.g., increased tortuosity in severe CTE)?
- [ ] Which features are redundant (highly correlated)?
- [ ] Are effect sizes large enough to be biologically meaningful?

**Next steps:** Feed significant features into the MIL framework (Phase 4).